# Rhetoric and Behavior After Party Switching in Brazil
## Replication Package - Comprehensive Analysis

**Author:** Arthur Gomes Nery  
**Date:** January 2026

---

### Table of Contents

| Part | Section | Content |
|------|---------|----------|
| **I** | 1.0 | Setup & Configuration |
| **I** | 1.1 | Data Loading |
| **I** | 1.2 | Sample Construction |
| **II** | 2.0 | Party Classifier Training |
| **II** | 2.1 | Main DML Event Study |
| **II** | 2.2 | Language vs. Voting Behavior |
| **III** | 3.1 | Robustness: Alternative Estimation Methods |
| **III** | 3.2 | Robustness: Classifier Performance |
| **III** | 3.3 | Robustness: Named Entity Removal |
| **III** | 3.4 | Robustness: Ideological Bloc Classification |
| **III** | 3.5 | Robustness: Embedding-Based Semantic Distance |
| **III** | 3.6 | Robustness: Placebo Test (Permutation) |
| **III** | 3.7 | Robustness: Alternative Time Windows |
| **III** | 3.8 | Robustness: Alternative Outcome Variable |
| **IV** | 4.1 | Heterogeneity: Within-Bloc vs Cross-Bloc |
| **IV** | 4.2 | Heterogeneity: Ideological Distance |
| **IV** | 4.3 | Heterogeneity: Career Experience |
| **IV** | 4.4 | Heterogeneity: Destination Party Size |
| **IV** | 4.5 | Heterogeneity: Switch Direction |
| **V** | 5.0 | Tables & Figures Export |

---
# PART I: DATA
---

### 1.0 Setup & Configuration

In [ ]:
import os
import pickle
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import stats
from scipy.spatial.distance import cosine
import statsmodels.api as sm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression, LassoCV, RidgeCV
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold
from sklearn.metrics.pairwise import cosine_distances

warnings.filterwarnings('ignore')
tqdm.pandas()
SEED = 42
np.random.seed(SEED)

@dataclass
class Config:
    text_col: str = 'text_level_2'  # normalized, keeps NER, not stemmed
    text_col_clean: str = 'text_no_ner' # For robustness check
    time_window_months: int = 12
    bimester_days: int = 60
    min_speeches_per_period: int = 10
    min_party_speeches: int = 50
    control_holdout_ratio: float = 0.20
    n_control_samples: int = 5
    tfidf_max_features: int = 5000
    tfidf_min_df: int = 5
    tfidf_max_df: float = 0.70
    n_partisan_words: int = 200
    dml_n_splits: int = 3
    n_permutations: int = 1000
    
    @property
    def window_days(self): return self.time_window_months * 30

CFG = Config()

# Paths
DATA_DIR = '../data/processed/'
RAW_DIR = '../data/raw/'
RESULTS_DIR = '../results/final_paper/'
PLOTS_DIR = os.path.join(RESULTS_DIR, 'figures/')
TABLES_DIR = os.path.join(RESULTS_DIR, 'tables/')

for d in [RESULTS_DIR, PLOTS_DIR, TABLES_DIR]:
    os.makedirs(d, exist_ok=True)

PATH_PANEL = os.path.join(DATA_DIR, 'data_panel.parquet')
PATH_HISTORY = os.path.join(RAW_DIR, 'deputies/deputy_migrations.csv')

# Visual settings
COLORS = {'main': '#2b7bba', 'control': '#0077BE', 'sig': '#2ECC71', 'nonsig': '#95A5A6'}
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 300, 'font.size': 11})

### 1.1 Data Loading

In [ ]:
# Load Panel Data
df = pd.read_parquet(PATH_PANEL).dropna(subset=['CAT', 'EST', CFG.text_col])
df['deputado_id'] = df['deputado_id'].astype(str)
df['idPartido'] = df['idPartido'].astype(str)
df['dataHoraInicio'] = pd.to_datetime(df['dataHoraInicio'], utc=True)

CAT_MAP = {0: 'Left', 1: 'Center', 2: 'Right'}
if pd.api.types.is_numeric_dtype(df['CAT']):
    df['CAT'] = df['CAT'].map(CAT_MAP)

# Load History
df_hist = pd.read_csv(PATH_HISTORY)
df_hist['deputado_id'] = df_hist['deputado_id'].astype(str)
df_hist['dataHora'] = pd.to_datetime(df_hist['dataHora'], utc=True)
df_hist['idPartido'] = df_hist['uriPartido'].str.split('/').str[-1]

# --- Dataset Overview Statistics ---
print("--- Dataset Overview ---")
overview_stats = pd.DataFrame([
    {'Metric': 'Total Speeches', 'Value': len(df)},
    {'Metric': 'Total Deputies', 'Value': df['deputado_id'].nunique()},
    {'Metric': 'Deputies with History', 'Value': df_hist['deputado_id'].nunique()}
])
print(overview_stats.to_string(index=False))

### 1.2 Sample Construction

In [ ]:
# 1. Extract Switch Events
switch_rows = df_hist[df_hist['descricaoStatus'] == 'Alteração de partido'].copy()

events = []
for dep_id, group in tqdm(switch_rows.groupby('deputado_id'), desc="Extracting events"):
    dep_hist = df_hist[df_hist['deputado_id'] == dep_id].sort_values('dataHora')
    for i, (_, row) in enumerate(group.sort_values('dataHora').iterrows()):
        before = dep_hist[dep_hist['dataHora'] < row['dataHora']]
        if len(before) == 0: continue
        old_pid, new_pid = before.iloc[-1]['idPartido'], row['idPartido']
        if old_pid == new_pid: continue
        events.append({'deputado_id': dep_id, 'switch_date': row['dataHora'],
                       'switch_number': i+1, 'old_party_id': str(old_pid), 'new_party_id': str(new_pid)})

df_events = pd.DataFrame(events)

# 2. Enrich with Ideology
def get_party_ideo(df, pid, date):
    sub = df[(df['idPartido']==str(pid)) & (df['dataHoraInicio']<=date)]
    if len(sub)==0: sub = df[df['idPartido']==str(pid)]
    return (sub.iloc[0]['EST'], sub.iloc[0]['CAT']) if len(sub)>0 else (None, None)

ideo_data = []
for _, row in tqdm(df_events.iterrows(), total=len(df_events), desc="Enriching ideology"):
    o_est, o_cat = get_party_ideo(df, row['old_party_id'], row['switch_date'])
    n_est, n_cat = get_party_ideo(df, row['new_party_id'], row['switch_date'])
    ideo_data.append({'old_EST': o_est, 'old_CAT': o_cat, 'new_EST': n_est, 'new_CAT': n_cat})

df_events = pd.concat([df_events, pd.DataFrame(ideo_data)], axis=1)
df_events['ideo_distance'] = abs(pd.to_numeric(df_events['new_EST'], errors='coerce') - 
                                  pd.to_numeric(df_events['old_EST'], errors='coerce'))
df_events['is_rightward'] = pd.to_numeric(df_events['new_EST'], errors='coerce') > pd.to_numeric(df_events['old_EST'], errors='coerce')
df_events['transition'] = df_events['old_CAT'] + ' -> ' + df_events['new_CAT']

# --- Switch Analysis Stats ---
print("\n--- Switch Event Analysis ---")
switch_stats = pd.DataFrame([
    {'Metric': 'Total Switch Events', 'Count': len(df_events)},
    {'Metric': 'Deputies Switching', 'Count': df_events['deputado_id'].nunique()},
    {'Metric': 'Events with Valid Ideology', 'Count': df_events.dropna(subset=['old_CAT','new_CAT']).shape[0]}
])
print(switch_stats.to_string(index=False))

In [ ]:
# 3. Experimental Partitioning
all_ids = df['deputado_id'].unique()
switcher_ids = df[df['party_change_count'] > 0]['deputado_id'].unique()
nonswitcher_ids = np.setdiff1d(all_ids, switcher_ids)
np.random.seed(SEED)
np.random.shuffle(nonswitcher_ids)

n_ctrl = int(len(nonswitcher_ids) * CFG.control_holdout_ratio)
control_ids, train_ids = nonswitcher_ids[:n_ctrl], nonswitcher_ids[n_ctrl:]
df_train = df[df['deputado_id'].isin(train_ids)].copy()

# --- Partition Stats ---
print("--- Experimental Partitioning ---")
part_stats = pd.DataFrame([
    {'Group': 'Train Deputies', 'Count': len(train_ids)},
    {'Group': 'Train Speeches', 'Count': len(df_train)},
    {'Group': 'Control Deputies', 'Count': len(control_ids)},
    {'Group': 'Switcher Deputies', 'Count': len(switcher_ids)}
])
print(part_stats.to_string(index=False))

In [ ]:
# 4. Voting Data Analysis & Event Study Build
PATH_VOTES = os.path.join(RAW_DIR, 'scrape_votes.parquet')
df_votes = pd.read_parquet(PATH_VOTES)
df_votes['deputado_id'] = df_votes['deputado_id'].astype(str)

# Handle date column
if 'dataVotacao' in df_votes.columns:
    df_votes['vote_date'] = pd.to_datetime(df_votes['dataVotacao'], utc=True)
elif 'dataHoraVoto' in df_votes.columns:
    df_votes['vote_date'] = pd.to_datetime(df_votes['dataHoraVoto'], utc=True)

# Build party timeline
deputy_party_timeline = {}
for deputy_id in tqdm(df_hist['deputado_id'].unique(), desc="Building Timelines"):
    affiliations = df_hist[df_hist['deputado_id'] == deputy_id].sort_values('dataHora')
    timeline = []
    for i, row in affiliations.iterrows():
        start = row['dataHora']
        end = affiliations.iloc[i+1]['dataHora'] if i < len(affiliations)-1 else pd.Timestamp('2030-01-01', tz='UTC')
        timeline.append({'start': start, 'end': end, 'party_id': row['idPartido']})
    deputy_party_timeline[deputy_id] = timeline

# Add party affiliation to votes
def get_party_at_date(deputy_id, date):
    if deputy_id not in deputy_party_timeline: return None
    for period in deputy_party_timeline[deputy_id]:
        if period['start'] <= date <= period['end']: return period['party_id']
    return None

df_votes['party_at_vote'] = df_votes.progress_apply(
    lambda r: get_party_at_date(r['deputado_id'], r['vote_date']), axis=1
)

# Calculate loyalty
def calc_loyalty(deputy_id, start_date, end_date, party_id):
    deputy_votes = df_votes[
        (df_votes['deputado_id'] == deputy_id) &
        (df_votes['vote_date'] >= start_date) & (df_votes['vote_date'] <= end_date)
    ]
    if len(deputy_votes) < 5: return np.nan
    
    vote_id_col = 'idVotacao' if 'idVotacao' in df_votes.columns else 'uriProposicao'
    party_votes = df_votes[
        (df_votes['party_at_vote'] == party_id) & (df_votes[vote_id_col].isin(deputy_votes[vote_id_col]))
    ]
    if len(party_votes) == 0: return np.nan
    
    party_position = party_votes.groupby(vote_id_col)['voto'].agg(
        lambda x: x.mode()[0] if len(x.mode()) > 0 else None
    )
    
    deputy_votes = deputy_votes.merge(party_position.to_frame('party_pos'), left_on=vote_id_col, right_index=True, how='left').dropna(subset=['party_pos'])
    if len(deputy_votes) < 5: return np.nan
    
    return (deputy_votes['voto'] == deputy_votes['party_pos']).mean()

voting_data = []
for _, switch in tqdm(df_events.iterrows(), total=len(df_events), desc="Calculating Loyalty"):
    deputy_id = switch['deputado_id']
    switch_date = pd.to_datetime(switch['switch_date'], utc=True)
    
    loyalty_pre = calc_loyalty(deputy_id, switch_date - pd.Timedelta(days=180), switch_date, switch['old_party_id'])
    loyalty_post = calc_loyalty(deputy_id, switch_date, switch_date + pd.Timedelta(days=180), switch['new_party_id'])
    
    voting_data.append({
        'deputado_id': deputy_id,
        'loyalty_pre_old': loyalty_pre,
        'loyalty_post_new': loyalty_post,
        'loyalty_increase': loyalty_post - loyalty_pre if pd.notna([loyalty_pre, loyalty_post]).all() else np.nan
    })

df_voting = pd.DataFrame(voting_data)

# Build Event Study Dataset
df_sw = df[df['deputado_id'].isin(df_events['deputado_id'].unique())].copy()
df_es = df_sw.merge(df_events[['deputado_id','switch_date','old_party_id','new_party_id',
                                'old_CAT','new_CAT','ideo_distance','is_rightward','switch_number']], on='deputado_id')

df_es['days_from_switch'] = (df_es['dataHoraInicio'].dt.tz_localize(None) - 
                              pd.to_datetime(df_es['switch_date']).dt.tz_localize(None)).dt.days
df_es = df_es[df_es['days_from_switch'].abs() <= 365].copy()

# --- Voting & Sample Stats ---
print("\n--- Event Study & Voting Loyalty ---")
vote_stats = pd.DataFrame([
    {'Metric': 'Event Study Sample', 'Value': len(df_es)},
    {'Metric': 'Total Votes Loaded', 'Value': len(df_votes)},
    {'Metric': 'Switchers with Voting Data', 'Value': len(df_voting.dropna(subset=['loyalty_increase']))}
])
print(vote_stats.to_string(index=False))

---
# PART II: MAIN RESULTS
---

### 2.0 Party Classifier Training

In [ ]:
tfidf_party = TfidfVectorizer(max_features=CFG.tfidf_max_features, min_df=CFG.tfidf_min_df,
                               max_df=CFG.tfidf_max_df, ngram_range=(1,2))
X_train = tfidf_party.fit_transform(df_train[CFG.text_col])
y_train = df_train['idPartido']

clf_party = LogisticRegression(class_weight='balanced', C=1.0, max_iter=500, n_jobs=-1, random_state=SEED)
clf_party.fit(X_train, y_train)

cv_scores = cross_val_score(clf_party, X_train, y_train, cv=5)
PARTY_CLASS_INDICES = {label: idx for idx, label in enumerate(clf_party.classes_)}

# --- Model Performance ---
print("--- Model Performance ---")
model_stats = pd.DataFrame([
    {'Metric': 'Model', 'Value': 'Logistic Regression'},
    {'Metric': 'CV Accuracy', 'Value': f"{cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})"},
    {'Metric': 'Training Samples', 'Value': len(y_train)}
])
print(model_stats.to_string(index=False))

### 2.1 Main DML Event Study

In [ ]:
# 1. Prediction Logic
X_es = tfidf_party.transform(df_es[CFG.text_col])
df_es['party_probs'] = list(clf_party.predict_proba(X_es))

def get_old_party_conf(row, ci):
    pid = str(row['old_party_id'])
    return row['party_probs'][ci[pid]] if pid in ci else np.nan

df_es['Y_confidence'] = df_es.apply(lambda r: get_old_party_conf(r, PARTY_CLASS_INDICES), axis=1)
df_es = df_es.dropna(subset=['Y_confidence'])

# --- Event Study Data Stats ---
print("--- Event Study Data ---")
es_stats = pd.DataFrame([
    {'Metric': 'Total Speech-Events', 'Value': len(df_es)},
    {'Metric': 'Rightward Switches', 'Value': df_es['is_rightward'].sum()},
    {'Metric': 'Avg Confidence', 'Value': f"{df_es['Y_confidence'].mean():.4f}"}
])
print(es_stats.to_string(index=False))

In [ ]:
# 2. DML Computation
bins = list(range(-180, 181, 30))
labels = [-6,-5,-4,-3,-2,-1,1,2,3,4,5,6]
df_es['month'] = pd.cut(df_es['days_from_switch'], bins=bins, labels=labels)
time_dummies = pd.get_dummies(df_es['month'], prefix='t')
if 't_-1' in time_dummies.columns:
    time_dummies = time_dummies.drop(columns=['t_-1'])

cov_cols = [c for c in ['gov_loyalty_12m','party_tenure_months'] if c in df_es.columns]
X_cov = df_es[cov_cols].fillna(0) if cov_cols else pd.DataFrame(index=df_es.index)
X_leg = pd.get_dummies(df_es['idLegislatura'], prefix='leg')

if 'topic_id' in df_es.columns:
    X_topic = pd.get_dummies(df_es['topic_id'], prefix='topic', drop_first=True)
else:
    X_topic = pd.DataFrame(index=df_es.index)

X = pd.concat([X_cov, X_leg, X_topic], axis=1)
Y = df_es['Y_confidence'].values
clusters = df_es['deputado_id'].values

kf = KFold(n_splits=CFG.dml_n_splits, shuffle=True, random_state=SEED)
learner_Y = HistGradientBoostingRegressor(max_iter=100, max_depth=5, random_state=SEED)
learner_D = HistGradientBoostingClassifier(max_iter=50, max_depth=3, random_state=SEED)

Y_pred = cross_val_predict(learner_Y, X, Y, cv=kf)
Y_resid = Y - Y_pred

D_resid = pd.DataFrame(index=df_es.index, columns=time_dummies.columns, dtype=float)
for col in tqdm(time_dummies.columns, desc="DML Residualizing"):
    D_pred = cross_val_predict(learner_D, X, time_dummies[col].values, cv=kf, method='predict_proba')[:,1]
    D_resid[col] = time_dummies[col].values - D_pred

X_final = sm.add_constant(D_resid)
results_dml = sm.OLS(Y_resid, X_final).fit(cov_type='cluster', cov_kwds={'groups': clusters})

# --- DML Model Summary ---
print("\n--- DML Model Summary ---")
print(f"N (speech-events):   {len(df_es):,}")
print(f"N (deputies):        {df_es['deputado_id'].nunique():,}")
print(f"N (clusters):        {len(np.unique(clusters)):,}")
print(f"R²:                  {results_dml.rsquared:.4f}")

In [ ]:
# 3. Pre-trends Test
pre_period_cols = [c for c in results_dml.params.index if c.startswith('t_-') and c != 'const']
pre_period_cols_sorted = sorted(pre_period_cols, key=lambda x: int(x.split('_')[1]))
beta_pre = results_dml.params[pre_period_cols_sorted].values
vcov_pre = results_dml.cov_params().loc[pre_period_cols_sorted, pre_period_cols_sorted].values

wald_stat = beta_pre @ np.linalg.inv(vcov_pre) @ beta_pre
f_stat = wald_stat / len(beta_pre)
df_numerator = len(beta_pre)
df_denominator = results_dml.df_resid
p_value_f = 1 - stats.f.cdf(f_stat, df_numerator, df_denominator)

# --- Pre-trends Result ---
print("--- Pre-trends Test ---")
print(f"F-statistic:         {f_stat:.3f}")
print(f"p-value:             {p_value_f:.4f}")
print(f"Degrees of freedom:  F({df_numerator}, {df_denominator})")

In [ ]:
# 4. Table 1 Output
print("\n" + "="*80)
print("TABLE 1: DML EVENT STUDY COEFFICIENTS")
print("="*80)

time_periods = [-6, -5, -4, -3, -2, 1, 2, 3, 4, 5, 6]
print("\n--- COEFFICIENTS FOR LATEX TABLE ---")
print("Period | Coef      | SE       | p-value  | CI Lower  | CI Upper")
print("-" * 70)

for t in time_periods:
    col_name = f't_{t}'
    if col_name in results_dml.params.index:
        coef = results_dml.params[col_name]
        se = results_dml.bse[col_name]
        pval = results_dml.pvalues[col_name]
        ci = results_dml.conf_int().loc[col_name]
        stars = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.1 else ''
        print(f"τ = {t:+2d} | {coef:9.4f}{stars:3s} | {se:8.4f} | {pval:8.3f} | {ci[0]:9.4f} | {ci[1]:9.4f}")

### 2.2 Language vs. Voting Behavior

In [ ]:
linguistic_effects = []
for deputy_id in df_events['deputado_id'].unique():
    dep_speeches = df_es[df_es['deputado_id'] == deputy_id]
    pre = dep_speeches[(dep_speeches['days_from_switch'] < 0) & (dep_speeches['days_from_switch'] >= -180)]['Y_confidence']
    post = dep_speeches[(dep_speeches['days_from_switch'] > 0) & (dep_speeches['days_from_switch'] <= 180)]['Y_confidence']
    if len(pre) >= 5 and len(post) >= 5:
        effect = post.mean() - pre.mean()
        linguistic_effects.append({'deputado_id': deputy_id, 'linguistic_effect': effect, 'abs_linguistic_effect': abs(effect)})

df_ling = pd.DataFrame(linguistic_effects)
df_corr = df_voting.merge(df_ling, on='deputado_id', how='inner')
df_corr = df_corr.dropna(subset=['loyalty_increase', 'abs_linguistic_effect'])

# --- Correlation Analysis Stats ---
print("--- Correlation Analysis ---")
corr_stats = pd.DataFrame([
    {'Metric': 'Switchers with Full Data', 'Value': len(df_corr)},
    {'Metric': 'Avg Linguistic Effect', 'Value': f"{df_corr['linguistic_effect'].mean():.4f}"}
])
print(corr_stats.to_string(index=False))

---
# PART III: ROBUSTNESS CHECKS
---

### 3.1 Robustness: Alternative Estimation Methods

In [ ]:
# Re-estimate Main DML using OLS and Lasso instead of Gradient Boosting
print("--- Robustness: Alternative Estimators ---")
estimators = {
    'LassoCV': (LassoCV(cv=5, random_state=SEED), LassoCV(cv=5, random_state=SEED)),
    'RidgeCV': (RidgeCV(cv=5), RidgeCV(cv=5))
}

for name, (model_y, model_d) in estimators.items():
    # Residualize Y
    Y_resid_alt = Y - cross_val_predict(model_y, X, Y, cv=kf)
    
    # Residualize D
    D_resid_alt = pd.DataFrame(index=df_es.index, columns=time_dummies.columns, dtype=float)
    for col in time_dummies.columns:
        D_resid_alt[col] = time_dummies[col] - cross_val_predict(model_d, X, time_dummies[col], cv=kf)
    
    # OLS
    res_alt = sm.OLS(Y_resid_alt, sm.add_constant(D_resid_alt)).fit(cov_type='cluster', cov_kwds={'groups': clusters})
    
    # Report key periods
    print(f"\nModel: {name}")
    for t in [1, 2, 3, 4, 5, 6]:
        if f't_{t}' in res_alt.params:
            print(f"τ = +{t} | Coef: {res_alt.params[f't_{t}']:.4f} | p-val: {res_alt.pvalues[f't_{t}']:.4f}")

### 3.2 Robustness: Classifier Performance

In [ ]:
# Filter Event Study to only high-confidence predictions (Top Decile of party distinguishability)
print("--- Robustness: High Confidence Sample ---")

# Calculate max probability for the assigned party in training
max_probs = np.max(clf_party.predict_proba(X_train), axis=1)
threshold = np.percentile(max_probs, 90)

print(f"Confidence Threshold (90th percentile): {threshold:.4f}")

# Filter ES
high_conf_ids = df_train[max_probs >= threshold]['deputado_id'].unique()
mask_conf = df_es['deputado_id'].isin(high_conf_ids)

if mask_conf.sum() > 100:
    res_conf = sm.OLS(Y_resid[mask_conf], X_final[mask_conf]).fit(cov_type='cluster', cov_kwds={'groups': clusters[mask_conf]})
    print(f"N (Subset): {mask_conf.sum()}")
    print(f"τ = +1 | Coef: {res_conf.params['t_1']:.4f} | p-val: {res_conf.pvalues['t_1']:.4f}")
else:
    print("Insufficient data for high-confidence subset.")

### 3.3 Robustness: Named Entity Removal

In [ ]:
print("--- Robustness: NER Removed ---")
if 'text_no_ner' in df.columns:
    print("Re-training classifier on anonymized text...")
    # Quick retraining cycle
    tfidf_ner = TfidfVectorizer(max_features=3000, min_df=5, max_df=0.7)
    X_train_ner = tfidf_ner.fit_transform(df_train['text_no_ner'])
    clf_ner = LogisticRegression(class_weight='balanced', C=1.0, max_iter=200, n_jobs=-1, random_state=SEED)
    clf_ner.fit(X_train_ner, y_train)
    
    # Re-predict ES
    X_es_ner = tfidf_ner.transform(df_es['text_no_ner'])
    probs_ner = clf_ner.predict_proba(X_es_ner)
    
    ner_indices = {l: i for i, l in enumerate(clf_ner.classes_)}
    Y_ner = df_es.apply(lambda r: r['party_probs'][ner_indices[str(r['old_party_id'])]] 
                        if str(r['old_party_id']) in ner_indices else np.nan, axis=1).dropna()
    
    # DML Simplified (OLS on residuals from main model, just swapping Y)
    # Note: Rigorous approach would re-residualize Y, here we show direct comparison for brevity
    common_idx = Y_ner.index.intersection(X_final.index)
    res_ner = sm.OLS(Y_ner.loc[common_idx], X_final.loc[common_idx]).fit(cov_type='cluster', cov_kwds={'groups': clusters[df_es.index.isin(common_idx)]})
    print(f"τ = +1 | Coef: {res_ner.params['t_1']:.4f} | p-val: {res_ner.pvalues['t_1']:.4f}")
else:
    print("Skipping: 'text_no_ner' column not found.")

### 3.4 Robustness: Ideological Bloc Classification

In [ ]:
print("--- Robustness: Ideological Blocs ---")
df_train['idBloc'] = df_train['CAT'].map({'Left': 0, 'Center': 1, 'Right': 2})
valid_blocs = df_train.dropna(subset=['idBloc'])

if len(valid_blocs) > 1000:
    # Train Bloc Classifier
    tfidf_bloc = TfidfVectorizer(max_features=3000, min_df=5)
    X_bloc = tfidf_bloc.fit_transform(valid_blocs[CFG.text_col])
    clf_bloc = LogisticRegression(class_weight='balanced', max_iter=200, n_jobs=-1, random_state=SEED)
    clf_bloc.fit(X_bloc, valid_blocs['idBloc'])
    
    # Predict
    X_es_bloc = tfidf_bloc.transform(df_es[CFG.text_col])
    probs_bloc = clf_bloc.predict_proba(X_es_bloc)
    
    # Y = Confidence in Old Bloc
    bloc_map = {'Left': 0, 'Center': 1, 'Right': 2}
    Y_bloc = []
    for i, r in df_es.iterrows():
        if r['old_CAT'] in bloc_map:
            Y_bloc.append(probs_bloc[i, bloc_map[r['old_CAT']]])
        else:
            Y_bloc.append(np.nan)
            
    Y_bloc = pd.Series(Y_bloc, index=df_es.index).dropna()
    idx_b = Y_bloc.index.intersection(X_final.index)
    
    res_bloc = sm.OLS(Y_bloc.loc[idx_b], X_final.loc[idx_b]).fit(cov_type='cluster', cov_kwds={'groups': clusters[df_es.index.isin(idx_b)]})
    print(f"τ = +1 | Coef: {res_bloc.params['t_1']:.4f} | p-val: {res_bloc.pvalues['t_1']:.4f}")
else:
    print("Insufficient bloc data.")

### 3.5 Robustness: Embedding-Based Semantic Distance

In [ ]:
print("--- Robustness: Embedding Distance ---")
# Calculate Cosine Distance between speech and Old Party Centroid in TF-IDF space

# 1. Compute Centroids
party_centroids = {}
for pid in df_train['idPartido'].unique():
    indices = df_train[df_train['idPartido'] == pid].index
    if len(indices) > 0:
        centroid = X_train[df_train.index.get_indexer(indices)].mean(axis=0)
        party_centroids[pid] = np.asarray(centroid)

# 2. Compute Distance for ES
dists = []
valid_indices = []
X_es_arr = X_es  # Sparse matrix

for i in range(X_es_arr.shape[0]):
    pid = str(df_es.iloc[i]['old_party_id'])
    if pid in party_centroids:
        # Cosine distance: 1 - cosine_similarity
        d = cosine_distances(X_es_arr[i], party_centroids[pid])
        dists.append(d[0][0])
        valid_indices.append(df_es.index[i])
    else:
        dists.append(np.nan)

Y_dist = pd.Series(dists, index=df_es.index).dropna()
idx_d = Y_dist.index.intersection(X_final.index)

res_dist = sm.OLS(Y_dist.loc[idx_d], X_final.loc[idx_d]).fit(cov_type='cluster', cov_kwds={'groups': clusters[df_es.index.isin(idx_d)]})
print("Outcome: Cosine Distance to Old Party (Expected positive coef for distancing)")
print(f"τ = +1 | Coef: {res_dist.params['t_1']:.4f} | p-val: {res_dist.pvalues['t_1']:.4f}")

### 3.6 Robustness: Placebo Test (Permutation)

In [ ]:
print("--- Robustness: Placebo Dates ---")
n_permutations = 100  # Reduced for runtime in reproduction
placebo_coefs = []

print(f"Running {n_permutations} permutations on t_1 coefficient...")
original_coef = results_dml.params['t_1']

for _ in tqdm(range(n_permutations), desc="Permuting"):
    # Shuffle time dummies relative to Y
    X_perm = X_final.sample(frac=1, random_state=None).reset_index(drop=True)
    X_perm.index = X_final.index
    
    try:
        res_perm = sm.OLS(Y_resid, X_perm).fit()
        placebo_coefs.append(res_perm.params['t_1'])
    except:
        continue

pval_perm = (np.abs(placebo_coefs) >= abs(original_coef)).mean()
print(f"Permutation p-value: {pval_perm:.4f}")

### 3.7 Robustness: Alternative Time Windows

In [ ]:
print("--- Robustness: Time Windows ---")
windows = [6, 9, 18] # Months

for w in windows:
    print(f"\nWindow: +/- {w} Months")
    mask = df_es['days_from_switch'].abs() <= (w * 30)
    if mask.sum() > 100:
        subset_Y = Y_resid[mask]
        subset_X = X_final.loc[mask]
        res_win = sm.OLS(subset_Y, subset_X).fit(cov_type='cluster', cov_kwds={'groups': clusters[mask]})
        print(f"N: {len(subset_Y)}")
        if 't_1' in res_win.params:
            print(f"τ = +1 | Coef: {res_win.params['t_1']:.4f} | p-val: {res_win.pvalues['t_1']:.4f}")
    else:
        print("Insufficient data.")

### 3.8 Robustness: Alternative Outcome Variable

In [ ]:
print("--- Robustness: New Party Probability ---")
# Y = Probability assigned to NEW party
Y_new = df_es.apply(lambda r: get_old_party_conf(r, PARTY_CLASS_INDICES), axis=1) # Reusing func logic but need new party ID

def get_new_party_conf(row, ci):
    pid = str(row['new_party_id'])
    return row['party_probs'][ci[pid]] if pid in ci else np.nan

Y_new = df_es.apply(lambda r: get_new_party_conf(r, PARTY_CLASS_INDICES), axis=1).dropna()
idx_n = Y_new.index.intersection(X_final.index)

res_new = sm.OLS(Y_new.loc[idx_n], X_final.loc[idx_n]).fit(cov_type='cluster', cov_kwds={'groups': clusters[df_es.index.isin(idx_n)]})
print("Outcome: Confidence in New Party (Expected positive coef for adaptation)")
print(f"τ = +1 | Coef: {res_new.params['t_1']:.4f} | p-val: {res_new.pvalues['t_1']:.4f}")

---
# PART IV: HETEROGENEITY ANALYSIS
---

### 4.1 Heterogeneity: Within-Bloc vs Cross-Bloc

In [ ]:
print("--- Heterogeneity: Ideological Bloc ---")
df_es['cross_bloc'] = (df_es['old_CAT'] != df_es['new_CAT']).astype(int)

# Interaction Model
X_het = X_final.copy()
for t in time_dummies.columns:
    X_het[f'{t}_x_Cross'] = X_het[t] * df_es.loc[X_het.index, 'cross_bloc']

res_bloc = sm.OLS(Y_resid, X_het).fit(cov_type='cluster', cov_kwds={'groups': clusters})
print(f"Interaction term (τ=1 * CrossBloc): {res_bloc.params.get('t_1_x_Cross', 0):.4f}")
print(f"P-value: {res_bloc.pvalues.get('t_1_x_Cross', 1):.4f}")

### 4.2 Heterogeneity: Ideological Distance

In [ ]:
print("--- Heterogeneity: Ideological Distance ---")
# Split by Median Distance
median_dist = df_es['ideo_distance'].median()
df_es['high_dist'] = (df_es['ideo_distance'] > median_dist).astype(int)

X_het_dist = X_final.copy()
for t in time_dummies.columns:
    X_het_dist[f'{t}_x_HighDist'] = X_het_dist[t] * df_es.loc[X_het_dist.index, 'high_dist']

res_dist_het = sm.OLS(Y_resid, X_het_dist).fit(cov_type='cluster', cov_kwds={'groups': clusters})
print(f"Median Distance: {median_dist:.2f}")
print(f"Interaction term (τ=1 * HighDist): {res_dist_het.params.get('t_1_x_HighDist', 0):.4f}")

### 4.3 Heterogeneity: Career Experience

In [ ]:
print("--- Heterogeneity: Career Tenure ---")
if 'party_tenure_months' in df_es.columns:
    high_tenure = (df_es['party_tenure_months'] > df_es['party_tenure_months'].median()).astype(int)
    X_het_ten = X_final.copy()
    for t in time_dummies.columns:
        X_het_ten[f'{t}_x_HighTenure'] = X_het_ten[t] * high_tenure.loc[X_het_ten.index]
        
    res_ten = sm.OLS(Y_resid, X_het_ten).fit(cov_type='cluster', cov_kwds={'groups': clusters})
    print(f"Interaction term (τ=1 * HighTenure): {res_ten.params.get('t_1_x_HighTenure', 0):.4f}")
else:
    print("Tenure variable missing.")

### 4.4 Heterogeneity: Destination Party Size

In [ ]:
print("--- Heterogeneity: Destination Party Size ---")
# Calculate size from training data
party_sizes = df_train['idPartido'].value_counts().to_dict()
df_es['new_party_size'] = df_es['new_party_id'].map(party_sizes)
median_size = df_es['new_party_size'].median()
df_es['is_large_party'] = (df_es['new_party_size'] > median_size).astype(int)

X_het_size = X_final.copy()
for t in time_dummies.columns:
    X_het_size[f'{t}_x_LargeParty'] = X_het_size[t] * df_es.loc[X_het_size.index, 'is_large_party']

res_size = sm.OLS(Y_resid, X_het_size).fit(cov_type='cluster', cov_kwds={'groups': clusters})
print(f"Interaction term (τ=1 * LargeParty): {res_size.params.get('t_1_x_LargeParty', 0):.4f}")

### 4.5 Heterogeneity: Switch Direction

In [ ]:
print("--- Heterogeneity: Left vs Right Switch ---")
X_het_dir = X_final.copy()
for t in time_dummies.columns:
    X_het_dir[f'{t}_x_Rightward'] = X_het_dir[t] * df_es.loc[X_het_dir.index, 'is_rightward'].astype(int)

res_dir = sm.OLS(Y_resid, X_het_dir).fit(cov_type='cluster', cov_kwds={'groups': clusters})
print(f"Interaction term (τ=1 * Rightward): {res_dir.params.get('t_1_x_Rightward', 0):.4f}")

---
# PART V: OUTPUT & EXPORT
---

In [ ]:
print("--- Exporting Results ---")

table_path = os.path.join(TABLES_DIR, 'dml_results.tex')
plot_path = os.path.join(PLOTS_DIR, 'event_study.png')

print(f"Saving Table 1 to: {table_path}")
with open(table_path, 'w') as f:
    f.write(results_dml.summary().as_latex())

print(f"Saving Event Study Plot to: {plot_path}")

# Construct plot data
plot_data = []
for t in [-6, -5, -4, -3, -2, 1, 2, 3, 4, 5, 6]:
    c = f't_{t}'
    if c in results_dml.params:
        plot_data.append({
            'time': t, 
            'coef': results_dml.params[c], 
            'lower': results_dml.conf_int().loc[c][0], 
            'upper': results_dml.conf_int().loc[c][1]
        })
plot_df = pd.DataFrame(plot_data)

fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(plot_df['time'], plot_df['coef'], 
            yerr=[plot_df['coef'] - plot_df['lower'], plot_df['upper'] - plot_df['coef']], 
            fmt='o', color=COLORS['main'], ecolor=COLORS['nonsig'], capsize=3)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(-0.5, color='gray', linewidth=0.8, linestyle=':')
ax.set_xlabel('Months from Switch')
ax.set_ylabel('Effect on Old Party Similarity')
ax.set_title('Event Study: Linguistic Adaptation')
plt.savefig(plot_path, bbox_inches='tight')
plt.close()

print("Paper replication package generation complete.")